In [10]:
# Week 1 - Linear Regression, Polynomial, Multicollinearity, Interaction
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
df = pd.read_csv("diabetes_012.csv")  
TARGET = "Diabetes_012"
y = df[TARGET].astype(float)
X = df.drop(columns=[TARGET])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear Regression
lr = LinearRegression().fit(X_train, y_train)
pred = lr.predict(X_test)
print("BASELINE")
print("R2 :", r2_score(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

# Polynomial Features
poly2 = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lr", LinearRegression())
]).fit(X_train, y_train)
pred_poly2 = poly2.predict(X_test)
print("\nPOLY DEG=2")
print("R2 :", r2_score(y_test, pred_poly2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_poly2)))

# Multicollinearity Check: Correlation 
num_cols = X.select_dtypes(include=[np.number]).columns
corr = X[num_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = (
    upper.stack()
         .reset_index()
         .rename(columns={"level_0": "feature1", "level_1": "feature2", 0: "abs_corr"})
         .sort_values("abs_corr", ascending=False)
)
print("\nTOP 5 CORRELATED PAIRS")
print(pairs.head())

# Multicollinearity Check: Simple VIF
def simple_vif(df_num):
    feats = df_num.columns.tolist()
    out = []
    for col in feats:
        X_i = df_num.drop(columns=[col]).values
        y_i = df_num[col].values
        if np.isclose(y_i.var(), 0):
            out.append((col, float("inf")))
            continue
        r2 = LinearRegression().fit(X_i, y_i).score(X_i, y_i)
        vif = float("inf") if r2 >= 1 else 1.0 / (1.0 - r2)
        out.append((col, vif))
    return pd.DataFrame(out, columns=["feature", "VIF"]).sort_values("VIF", ascending=False)

vif_df = simple_vif(X[num_cols])
print("\nTOP 5 VIF VALUES")
print(vif_df.head())

# Add One Interaction (BMI x PhysActivity) 
X2 = X.copy()
if {"BMI","PhysActivity"}.issubset(X2.columns):
    X2["BMI_x_Phys"] = X2["BMI"] * X2["PhysActivity"]

    X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.2, random_state=42)
    lr2 = LinearRegression().fit(X2_train, y2_train)
    pred2 = lr2.predict(X2_test)
    print("\nINTERACTION (BMI x PhysActivity)")
    print("R2 :", r2_score(y2_test, pred2))
    print("RMSE:", np.sqrt(mean_squared_error(y2_test, pred2)))
else:
    print("\nColumns 'BMI' or 'PhysActivity' not found. Change them to real column names in your dataset.")

# Summary (
summary = {
    "baseline_R2": r2_score(y_test, pred),
    "baseline_RMSE": float(np.sqrt(mean_squared_error(y_test, pred))),
    "poly2_R2": r2_score(y_test, pred_poly2),
    "poly2_RMSE": float(np.sqrt(mean_squared_error(y_test, pred_poly2))),
    "interaction_R2": r2_score(y2_test, pred2) if "BMI_x_Phys" in X2 else None,
    "interaction_RMSE": float(np.sqrt(mean_squared_error(y2_test, pred2))) if "BMI_x_Phys" in X2 else None,
    "top_corr_pair": (pairs.iloc[0]["feature1"], pairs.iloc[0]["feature2"], float(pairs.iloc[0]["abs_corr"])) if not pairs.empty else None,
    "top_vif": (vif_df.iloc[0]["feature"], float(vif_df.iloc[0]["VIF"])) if not vif_df.empty else None,
}
print(summary)


BASELINE
R2 : 0.17330978646058792
RMSE: 0.6322608048598514

POLY DEG=2
R2 : 0.20258668422985848
RMSE: 0.6209642580080663

TOP 5 CORRELATED PAIRS
      feature1  feature2  abs_corr
183    GenHlth  PhysHlth  0.524364
195   PhysHlth  DiffWalk  0.478417
184    GenHlth  DiffWalk  0.456920
209  Education    Income  0.449106
188    GenHlth    Income  0.370014

TOP 5 VIF VALUES
     feature       VIF
13   GenHlth  1.795892
15  PhysHlth  1.623288
16  DiffWalk  1.533902
20    Income  1.503931
18       Age  1.349994

INTERACTION (BMI x PhysActivity)
R2 : 0.1740196654158621
RMSE: 0.6319892853559604

SUMMARY
{'baseline_R2': 0.17330978646058792, 'baseline_RMSE': 0.6322608048598514, 'poly2_R2': 0.20258668422985848, 'poly2_RMSE': 0.6209642580080663, 'interaction_R2': 0.1740196654158621, 'interaction_RMSE': 0.6319892853559604, 'top_corr_pair': ('GenHlth', 'PhysHlth', 0.524363643849337), 'top_vif': ('GenHlth', 1.7958923562116744)}


In [11]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score

# 1) Load data
df = pd.read_csv("diabetes_012.csv")
y = df["Diabetes_012"].astype(float)
X = df.drop(columns=["Diabetes_012"])

# 2) Split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# 3) CV settings + alpha grid (tiny & simple)
cv5 = KFold(n_splits=5, shuffle=True, random_state=42)
alphas = np.logspace(-3, 3, 13)   # 0.001 ... 1000 (13 values)
l1_ratios = [0.2, 0.5, 0.8]       # for Elastic Net

# 4) Lasso (feature selection)
lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(alphas=alphas, cv=cv5, max_iter=20000))
]).fit(Xtr, ytr)
pred_l = lasso.predict(Xte)
coef_l = lasso.named_steps["model"].coef_

# 5) Ridge (shrinkage)
ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(alphas=alphas, cv=cv5))
]).fit(Xtr, ytr)
pred_r = ridge.predict(Xte)

# 6) Elastic Net (mix of both)
enet = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(alphas=alphas, l1_ratio=l1_ratios, cv=cv5, max_iter=20000))
]).fit(Xtr, ytr)
pred_e = enet.predict(Xte)
coef_e = enet.named_steps["model"].coef_

# 7) Print simple results
rmse = lambda y, p: float(np.sqrt(mean_squared_error(y, p)))
print("\nLASSO  -> alpha* =", lasso.named_steps["model"].alpha_,
      "| R2:", r2_score(yte, pred_l), "| RMSE:", rmse(yte, pred_l),
      "| nonzero:", int(np.sum(coef_l != 0)))

print("RIDGE  -> alpha* =", ridge.named_steps["model"].alpha_,
      "| R2:", r2_score(yte, pred_r), "| RMSE:", rmse(yte, pred_r))

print("ELNET  -> alpha* =", enet.named_steps["model"].alpha_,
      " l1_ratio* =", enet.named_steps["model"].l1_ratio_,
      "| R2:", r2_score(yte, pred_e), "| RMSE:", rmse(yte, pred_e),
      "| nonzero:", int(np.sum(coef_e != 0)))

# 8) Tiny summary (easy to copy)
summary = {
    "lasso_alpha": float(lasso.named_steps["model"].alpha_),
    "lasso_R2": float(r2_score(yte, pred_l)),
    "lasso_RMSE": rmse(yte, pred_l),
    "lasso_nonzero": int(np.sum(coef_l != 0)),
    "ridge_alpha": float(ridge.named_steps["model"].alpha_),
    "ridge_R2": float(r2_score(yte, pred_r)),
    "ridge_RMSE": rmse(yte, pred_r),
    "enet_alpha": float(enet.named_steps["model"].alpha_),
    "enet_l1_ratio": float(enet.named_steps["model"].l1_ratio_),
    "enet_R2": float(r2_score(yte, pred_e)),
    "enet_RMSE": rmse(yte, pred_e),
    "enet_nonzero": int(np.sum(coef_e != 0)),
}
print("\nSUMMARY:", summary)



LASSO  -> alpha* = 0.001 | R2: 0.1732625700161402 | RMSE: 0.6322788604025606 | nonzero: 20
RIDGE  -> alpha* = 316.22776601683796 | R2: 0.1733079337005239 | RMSE: 0.6322615133640127
ELNET  -> alpha* = 0.001  l1_ratio* = 0.5 | R2: 0.17329587441938632 | RMSE: 0.6322661248704293 | nonzero: 20

SUMMARY: {'lasso_alpha': 0.001, 'lasso_R2': 0.1732625700161402, 'lasso_RMSE': 0.6322788604025606, 'lasso_nonzero': 20, 'ridge_alpha': 316.22776601683796, 'ridge_R2': 0.1733079337005239, 'ridge_RMSE': 0.6322615133640127, 'enet_alpha': 0.001, 'enet_l1_ratio': 0.5, 'enet_R2': 0.17329587441938632, 'enet_RMSE': 0.6322661248704293, 'enet_nonzero': 20}
